# Food Detection System using Images

## 1️⃣ IMPORTS

In [15]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

ModuleNotFoundError: No module named 'tensorflow'

## 2️⃣ ENABLE MIXED PRECISION

In [16]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

ModuleNotFoundError: No module named 'tensorflow'

## 3️⃣ CONFIGURATION

In [17]:
DATA_DIR = "/kaggle/input/food41/images/"
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

NameError: name 'tf' is not defined

## 4️⃣ LOAD DATASET (70% TRAIN / 15% VAL / 15% TEST)

In [18]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Number of classes:", num_classes)

temp_batches = tf.data.experimental.cardinality(temp_ds).numpy()
val_size = temp_batches // 2

val_ds = temp_ds.take(val_size)
test_ds = temp_ds.skip(val_size)

NameError: name 'tf' is not defined

## 5️⃣ DATA AUGMENTATION

In [19]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

NameError: name 'tf' is not defined

## 6️⃣ OPTIMIZE DATA PIPELINE

In [20]:
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

NameError: name 'train_ds' is not defined

## 7️⃣ BUILD MODEL (MobileNetV2 Transfer Learning)

In [5]:
base_model = tf.keras.applications.EfficientNetB2(
    input_shape=(256,256,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

inputs = layers.Input(shape=(256,256,3))

x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    num_classes,
    activation='softmax',
    dtype='float32'
)(x)

model = tf.keras.Model(inputs, outputs)

NameError: name 'tf' is not defined

## 8️⃣ COMPILE MODEL

In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=True)
    ]
)

model.summary()

NameError: name 'model' is not defined

## 9️⃣ CALLBACKS

In [7]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-6
    ),
    ModelCheckpoint(
        "best_mobilenetv2_food101.keras",
        save_best_only=True,
        monitor='val_loss'
    )
]

NameError: name 'EarlyStopping' is not defined

## 🔟 TRAIN MODEL

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

NameError: name 'model' is not defined

## 1️⃣1️⃣ FINAL EVALUATION ON TEST SET

In [9]:
test_loss, test_acc, test_auc = model.evaluate(test_ds)

print("Test Accuracy:", test_acc)
print("Test AUC:", test_auc)

NameError: name 'model' is not defined

## 1️⃣2️⃣ DETAILED METRICS

In [10]:
y_true = []
y_pred = []
y_pred_probs = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_probs = np.array(y_pred_probs)

NameError: name 'test_ds' is not defined

### 📊 Classification Report

In [11]:
print(classification_report(y_true, y_pred, target_names=class_names))

NameError: name 'classification_report' is not defined

### 📊 Confusion Matrix

In [12]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

NameError: name 'confusion_matrix' is not defined

### 📊 ROC-AUC (Multi-class)

In [13]:
y_true_bin = label_binarize(y_true, classes=range(num_classes))
roc_auc = roc_auc_score(y_true_bin, y_pred_probs, multi_class='ovr')

print("ROC-AUC Score:", roc_auc)

NameError: name 'label_binarize' is not defined

### 💾 SAVE ARTIFACTS

In [14]:
model.save("food101_mobilenetv2_final.h5")
model.save("food101_mobilenetv2_final.keras")

with open("class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)

print("Model and artifacts saved successfully.")

NameError: name 'model' is not defined